# ICLR 2027 — Next Experiments: Student A
## Geometry-Opportunity, Online-Gain Capture, and Controlled Held-Out Generalization
**Project**: *Geometry Determines Acceleration in Phase Retrieval* (ICLR 2027)  
**Author**: Student A  
**Package**: `Student_A_New` / `student_A_next_handoff_bundle.zip`

### Nhiệm vụ trọng tâm của Student A (Section 3, 4, 5, 6 - Handoff Guide):
1. **A1 / E6-A: Sổ cái chính tắc E6 (`e6_case_ledger.csv`)** — Tích hợp kết quả co cụm đóng băng từ H2 và Online Horizon Audit tại chân trời chuẩn $H=180$.
2. **A2 / E6-B: Phân tích Dư địa Hình học & Tỷ lệ Thu hồi Oracle Gain** — Ước lượng $\Delta_{\rm oracle}^{\rm obs}$, $\Delta_{\rm online}^{\rm obs}$, $R_{\rm capture}$, tính khoảng tin cậy Bootstrap 95%, tương quan Pearson/Spearman và vẽ 5 biểu đồ khoa học chuẩn (Figure A1a, A1b, A1c, A2, A3).
3. **A3 / E6-C: Thử nghiệm Tổng quát hóa Kiểm soát (Held-Out Geometry Generalization)** — Đóng băng trước 36 cấu hình ngẫu nhiên kèm mã băm SHA-256; chạy song song 5 thuật toán (GLA, FGLA-opt, Oracle minimax, Oracle conservative, Online `dynamic_envelope`).
4. **A4 / Gate A: Thẩm định Toàn vẹn & Xuất Bảng Công bố** — Kiểm tra 100% ràng buộc phổ và phân cấp tốc độ; xuất bảng LaTeX/CSV và đóng gói bàn giao `student_A_next_handoff_bundle.zip`.

---
*Notebook này được thiết kế **Zero-Upload**: Chứa sẵn toàn bộ mã nguồn `geometry_gla`, dữ liệu đóng băng H2 và script thực nghiệm dưới dạng nhúng tất định. Có thể chạy độc lập trên Kaggle (CPU/GPU) hoặc máy cá nhân/Server.*


In [ ]:
# [Cell 0] Khởi tạo Môi trường & Giải nén Tự động (Zero-Upload Self-Extractor)
import os, sys, base64, io, zipfile, importlib
from pathlib import Path

WORKING_DIR = Path("YOUR_WORKING_PATH") if Path("YOUR_WORKING_PATH").exists() else Path(".").resolve()
print(f"[INIT] Working directory: {WORKING_DIR}")

EMBEDDED_ZIP_B64 = """..."""

def setup_environment(target_dir):
    raw_bytes = base64.b64decode(EMBEDDED_ZIP_B64)
    with zipfile.ZipFile(io.BytesIO(raw_bytes), "r") as zf:
        for member in zf.infolist():
            norm_name = member.filename.replace("\\", "/").lstrip("/")
            out_file = target_dir / norm_name
            if member.is_dir() or norm_name.endswith("/"):
                out_file.mkdir(parents=True, exist_ok=True)
            else:
                out_file.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, open(out_file, "wb") as dst:
                    dst.write(src.read())
    print(f"[OK] Unpacked core codebase and frozen data into: {target_dir}")

# Extract if not already fully extracted
if not (WORKING_DIR / "src" / "geometry_gla").exists() or not (WORKING_DIR / "data" / "frozen" / "h2_multi.json").exists():
    setup_environment(WORKING_DIR)
else:
    print("[INFO] Codebase already detected locally.")

# Ensure sys.path points to necessary roots
for p in [WORKING_DIR, WORKING_DIR / "src", WORKING_DIR / "scripts"]:
    p_str = str(p.resolve())
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

import numpy as np
import scipy
import matplotlib.pyplot as plt
import geometry_gla
print(f"[OK] geometry_gla verified at: {geometry_gla.__file__}")
print(f"[OK] Environment: Python {sys.version.split()[0]}, NumPy {np.__version__}, SciPy {scipy.__version__}")


### Bước A1 / E6-A: Xây dựng Sổ cái Thực nghiệm Chính tắc E6
Tích hợp các kết quả co cụm đóng băng từ H2 và Online Horizon Audit tại chân trời kiểm soát $H=180$.
Sổ cái tính toán các đại lượng dẫn xuất theo đúng quy chuẩn Section 4 của Paper:
- Độ rộng phổ: $W = \lambda_+ - \lambda_-$
- Khoảng cách ngang: $G = 1 - \lambda_+$
- Dư địa Oracle dự đoán: $\Delta_{\rm oracle}^{\rm pred} = q_{\rm FGLA}^{\rm pred} - q_{\rm Oracle}^{\rm pred}$
- Dư địa Oracle quan sát: $\Delta_{\rm oracle}^{\rm obs} = q_{\rm FGLA}^{\rm obs} - q_{\rm Oracle}^{\rm obs}$
- Lợi thế Online quan sát: $\Delta_{\rm online}^{\rm obs} = q_{\rm FGLA}^{\rm obs} - q_{\rm Online}^{\rm obs}$
- Tỷ lệ thu nhận Oracle Gain: $R_{\rm capture} = \frac{\Delta_{\rm online}^{\rm obs}}{\Delta_{\rm oracle}^{\rm obs}}$ (khi $\Delta_{\rm oracle}^{\rm obs} > 10^{-8}$).


In [ ]:
# [Cell 1] Thực thi Bước A1: Build E6 Canonical Ledger
import pandas as pd
from scripts.01_build_e6_ledger import build_ledger, DEFAULT_H2_PATHS, DEFAULT_HZ_PATHS, find_file

# ==============================================================================
# CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU ĐÓNG BĂNG (TÙY CHỌN - ĐỂ NONE ĐỂ TỰ ĐỘNG TÌM KIẾM)
# ==============================================================================
YOUR_H2_PATH = None
YOUR_HORIZON_PATH = None
path_h2 = YOUR_H2_PATH
path_horizon = YOUR_HORIZON_PATH

h2_file = find_file(path_h2, [WORKING_DIR / "data" / "frozen" / "h2_multi.json"] + DEFAULT_H2_PATHS)
hz_file = find_file(path_horizon, [WORKING_DIR / "data" / "frozen" / "horizon_audit.json"] + DEFAULT_HZ_PATHS)
out_dir = WORKING_DIR / "results" / "e6_geometry_opportunity"

rows = build_ledger(h2_file, hz_file, out_dir, horizon_val=180, denom_tol=1e-8)
df_ledger = pd.DataFrame(rows)

print(f"\nTổng số trường hợp trong Sổ cái E6: {len(df_ledger)} case")
print(f"Số trường hợp có dữ liệu Online hợp lệ tại H=180: {df_ledger['q_online_obs'].notna().sum()} case")
display(df_ledger[['case_id', 'lambda_minus', 'lambda_plus', 'q_fgla_obs', 'q_oracle_obs', 'q_online_obs', 'delta_oracle_obs', 'delta_online_obs', 'r_capture']].head(10))


### Bước A2 / E6-B: Phân tích Dư địa Hình học & Biểu đồ Khoa học Chuẩn
Tính toán thống kê mô tả phi tham số (trung vị, IQR, quantile, Bootstrap 95% CI) và hệ số tương quan Pearson/Spearman.
Xuất và hiển thị trực tiếp 5 đồ thị chuẩn của Student A:
- **Figure A1a, A1b, A1c**: Quan hệ giữa Dư địa Oracle $\Delta_{\rm oracle}^{\rm obs}$ với $\lambda_+$, $1 - \lambda_+$, và $\lambda_+ - \lambda_-$.
- **Figure A2 (Đồ thị trung tâm)**: Đối sánh $x = \Delta_{\rm oracle}^{\rm obs}$ và $y = \Delta_{\rm online}^{\rm obs}$ kèm đường định danh $y=x$, đường $y=0$, hồi quy qua gốc tọa độ, hồi quy tuyến tính thường và đánh dấu minh bạch các trường hợp Online thua FGLA.
- **Figure A3**: Tỷ lệ thu hồi $R_{\rm capture}$ theo Dư địa Oracle $\Delta_{\rm oracle}^{\rm obs}$.


In [ ]:
# [Cell 2] Thực thi Bước A2: Analyze Geometry Opportunity & Display Figures
from scripts.02_analyze_geometry_opportunity import analyze_opportunity
from scripts.common import load_json
from IPython.display import Image, display

ledger_path = out_dir / "e6_case_ledger.csv"
summary_path = out_dir / "e6_summary.json"

summary = analyze_opportunity(ledger_path, summary_path, out_dir / "figures", denom_tol=1e-8)

print("\n" + "="*70)
print("KẾT QUẢ PHÂN TÍCH DƯ ĐỊA HÌNH HỌC & ONLINE GAIN (E6-B):")
print("="*70)
print(f"1. Dư địa Oracle quan sát (Delta_oracle^obs):")
print(f"   - Trung vị: {summary['observed_oracle_opportunity']['median']:.4f}")
print(f"   - IQR:      [{summary['observed_oracle_opportunity']['iqr'][0]:.4f}, {summary['observed_oracle_opportunity']['iqr'][1]:.4f}]")
print(f"2. Lợi thế Online quan sát (Delta_online^obs):")
print(f"   - Trung vị: {summary['observed_online_gain']['median']:.4f}")
print(f"   - Tỷ lệ thắng FGLA: {summary['oracle_gain_capture']['frac_gt_0']*100:.1f}%")
print(f"3. Tỷ lệ Thu hồi Oracle Gain (R_capture):")
print(f"   - Trung vị: {summary['oracle_gain_capture']['median']:.4f} (Bootstrap 95% CI: [{summary['oracle_gain_capture']['bootstrap95_median'][0]:.4f}, {summary['oracle_gain_capture']['bootstrap95_median'][1]:.4f}])")
print(f"   - Tỷ lệ R_capture >= 0.50: {summary['oracle_gain_capture']['frac_ge_0p5']*100:.1f}%")
print(f"   - Tỷ lệ R_capture >= 0.80: {summary['oracle_gain_capture']['frac_ge_0p8']*100:.1f}%")
print(f"4. Tương quan và Hồi quy:")
print(f"   - Pearson r:    {summary['correlation']['pearson']:.4f} (CI: [{summary['correlation']['pearson_bootstrap95_ci'][0]:.4f}, {summary['correlation']['pearson_bootstrap95_ci'][1]:.4f}])")
print(f"   - Spearman rho: {summary['correlation']['spearman']:.4f} (CI: [{summary['correlation']['spearman_bootstrap95_ci'][0]:.4f}, {summary['correlation']['spearman_bootstrap95_ci'][1]:.4f}])")
print(f"   - Hồi quy qua gốc tọa độ: slope = {summary['regression']['through_origin_slope']:.4f}")
print(f"   - Hồi quy thường: y = {summary['regression']['ordinary_slope']:.4f}x + ({summary['regression']['ordinary_intercept']:.4f})")

# Hiển thị các hình vẽ
fig_dir = out_dir / "figures"
print("\n--- Figure A2: Đồ thị trung tâm (Online Gain vs Oracle Opportunity) ---")
display(Image(filename=str(fig_dir / "A2_online_vs_oracle_gain.png")))

print("\n--- Figure A3: Tỷ lệ thu hồi R_capture theo Dư địa Oracle ---")
display(Image(filename=str(fig_dir / "A3_gain_capture_vs_oracle.png")))

print("\n--- Figure A1a, A1b, A1c: Dư địa Oracle theo Phổ Hình học ---")
display(Image(filename=str(fig_dir / "A1a_oracle_vs_lambda_plus.png")))
display(Image(filename=str(fig_dir / "A1b_oracle_vs_gap.png")))
display(Image(filename=str(fig_dir / "A1c_oracle_vs_width.png")))


### Bước A3 / E6-C: Thử nghiệm Tổng quát hóa Kiểm soát (Held-Out Geometry Generalization)
Quy tắc bất khả xâm phạm của Handoff Guide:
1. Đóng băng cấu hình 36 trường hợp mới vào `configs/e6_heldout_manifest_frozen.json` và kiểm tra mã băm SHA-256 trước khi chạy.
2. Không tinh chỉnh siêu tham số, không chạy lại để chọn trường hợp thuận lợi.
3. Chạy chính xác 5 phương pháp:
   - **GLA** chuẩn (Fixed $\alpha=0, \beta=0, \gamma=1$)
   - **FGLA-opt** tối ưu một đầu ($q = 1 - \sqrt{1-\lambda_+}$)
   - **Oracle minimax** hai đầu tối ưu toàn khoảng ($[\lambda_-, \lambda_+]$)
   - **Oracle conservative** bảo thủ ($[0, \lambda_+]$)
   - **Online safe** đóng băng (`dynamic_envelope`, $p=3, K=8$, margin=0.0)
4. Nếu gặp trường hợp không chính quy (non-regular neutral modes hoặc rank deficient), đánh dấu `skipped` kèm lý do cụ thể, tuyệt đối không thay thế ngầm.


In [ ]:
# [Cell 3] Thực thi Bước A3: Đóng băng & Chạy Thực nghiệm Held-Out
from scripts.03_freeze_heldout_grid import freeze_heldout_grid
from scripts.04_run_heldout_experiments import run_heldout_suite

spec_p = WORKING_DIR / "configs" / "heldout_grid_spec.json"
manifest_p = WORKING_DIR / "configs" / "e6_heldout_manifest_frozen.json"
sha_p = WORKING_DIR / "configs" / "heldout_manifest.sha256"

# 1. Đóng băng manifest và xác minh mã SHA256
manifest, digest = freeze_heldout_grid(spec_p, manifest_p, sha_p)
print(f"Xác minh SHA256 Held-Out Manifest: {digest}")

# 2. Chạy bộ thí nghiệm held-out
held_rows, held_summary = run_heldout_suite(manifest_p, out_dir, n_iter=180)
df_heldout = pd.DataFrame(held_rows)

print("\n" + "="*70)
print("KẾT QUẢ TỔNG QUÁT HÓA TRÊN TẬP HELD-OUT (E6-C):")
print("="*70)
print(f"Tổng số trường hợp: {held_summary['n_cases_total']} ({held_summary['n_cases_valid']} hợp lệ, {held_summary['n_cases_skipped']} skipped theo chuẩn)")
print(f"Dư địa Oracle trung vị: {held_summary['oracle_opportunity']['median']:.4f}")
print(f"Lợi thế Online trung vị: {held_summary['online_gain']['median']:.4f}")
print(f"Tỷ lệ Online thắng FGLA: {held_summary['online_gain']['win_rate_vs_fgla']*100:.1f}%")
print(f"Tỷ lệ thu hồi R_capture trung vị: {held_summary['capture']['median']:.4f}")
print(f"Tỷ lệ R_capture >= 0.50: {held_summary['capture']['frac_ge_0p5']*100:.1f}%")

display(df_heldout[['case_id', 'N', 'window', 'W', 'hop', 'lambda_plus', 'q_fgla_obs', 'q_oracle_obs', 'q_online_obs', 'delta_online_obs', 'r_capture', 'online_beats_fgla']].head(10))


### Bước A4 / Gate A: Thẩm định Toàn vẹn Khoa học & Bảng Số liệu Công bố
- **Thẩm định tự động (`integrity_report.json`)**: Kiểm tra không có giá trị riêng âm bất hợp pháp, giá trị $q$ trong giới hạn toán học, tuân thủ chặt chẽ phân cấp tốc độ $q_{\rm oracle} \le q_{\rm fgla} \le q_{\rm gla}$, không có mã case bị trùng lặp.
- **Bảng công bố (`table_e6_summary.tex` / `.csv`)**: Định dạng chuẩn bài báo ICLR 2027 đối sánh giữa tập chính tắc và tập tổng quát hóa.


In [ ]:
# [Cell 4] Thực thi Bước A4: Thẩm định Khoa học & Xuất Bảng Công bố
from scripts.05_validate_results import run_validation
from scripts.06_make_tables import make_tables

# 1. Thẩm định toàn vẹn
report = run_validation(out_dir / "e6_case_ledger.csv", out_dir / "e6_heldout_ledger.csv", out_dir)
print(f"\nKết quả Thẩm định Toàn vẹn: PASS = {report['pass']} (Số lỗi: {report['n_issues']} trên {report['total_cases_audited']} case)")

# 2. Tạo bảng công bố LaTeX & CSV
tex_file, csv_file = make_tables(out_dir / "e6_summary.json", out_dir / "e6_heldout_summary.json", out_dir)
df_table = pd.read_csv(csv_file)
display(df_table)

print("\nNội dung Bảng LaTeX (table_e6_summary.tex):")
print(tex_file.read_text(encoding='utf-8'))


### Bước A5 / Gate A: Báo cáo Tổng kết Tiếng Việt & Đóng gói Bàn giao (Handoff Bundle)
Tự động tạo báo cáo khoa học `STUDENT_A_NEXT_REPORT.md` bằng tiếng Việt theo phong cách chuẩn mực của nhóm nghiên cứu và đóng gói toàn bộ artifacts vào `student_A_next_handoff_bundle.zip` để tải về hoặc nộp cho Thầy / chuyển tiếp cho Student B.


In [ ]:
# [Cell 5] Xuất Báo cáo Tiếng Việt & Đóng gói Bàn giao
from scripts.07_generate_student_a_report import generate_report
from run_all_student_a_new import package_handoff_bundle, write_environment_metadata
from IPython.display import Markdown

# Tạo báo cáo khoa học
report_file = generate_report(out_dir / "e6_summary.json", out_dir / "e6_heldout_summary.json", out_dir / "integrity_report.json", out_dir)

# Ghi metadata môi trường và đóng gói ZIP
write_environment_metadata()
package_handoff_bundle()

bundle_zip = WORKING_DIR / "student_A_next_handoff_bundle.zip"
print(f"\n[GATE A HOÀN TẤT] File bàn giao sẵn sàng tại: {bundle_zip} ({bundle_zip.stat().st_size / 1024:.1f} KB)")

# Hiển thị tóm tắt báo cáo
display(Markdown((out_dir / "STUDENT_A_NEXT_REPORT.md").read_text(encoding='utf-8')[:3500] + "\n\n*(Đã cắt bớt để hiển thị - xem chi tiết tại STUDENT_A_NEXT_REPORT.md)*"))
